<a href="https://colab.research.google.com/github/rist-kobe/HPC-Programming/blob/main/Tuning/sample_code/13_polynomial/13_polynomial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

Install GNU Fortran and NVIDIA HPC SDK (optional for C-only examples; can take ~30 min)

In [ ]:
!sudo apt-get update -y
!sudo apt-get install -y build-essential gfortran curl gnupg

# Optional: install NVIDIA HPC SDK (large download, not needed for C-only examples).
# Uncomment the following lines to install it.
#!curl -fsSL https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
#!echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' | sudo tee /etc/apt/sources.list.d/nvhpc.list
#!sudo apt-get update -y
#!sudo apt-get install -y nvhpc-22-7-cuda-multi

import glob, os
nvhpc_bins = sorted(glob.glob('/opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin'), key=lambda p: tuple(int(x) for x in p.split('/Linux_x86_64/')[1].split('/')[0].replace('-', '.').split('.')), reverse=True)
if nvhpc_bins:
    nvhpc_bin = nvhpc_bins[0]
    current_path = os.environ.get('PATH', '')
    if nvhpc_bin not in current_path.split(':'):
        os.environ['PATH'] = nvhpc_bin + (':' + current_path if current_path else '')
    print('Using NVIDIA HPC SDK:', nvhpc_bin)
else:
    print('NVIDIA HPC SDK not found (optional; not needed for C-only examples).')


Clone the repository and change to the `13_polynomial` directory.

In [ ]:
%cd /content
!rm -rf HPC-Programming
!git clone https://github.com/rist-kobe/HPC-Programming.git
%cd HPC-Programming/Tuning/sample_code/13_polynomial
!ls


# Comparison between different implementations of a compute-bound kernel, n-th order polynomial
* Author:       Yukihiro Ota (yota@rist.or.jp)
* Last update:  30th Jan., 2024

## Directory tree

In [ ]:
!src/
!   + c/ 
!   |           low-level c code
!   + cpp/  
!   |           c-like low-level code with C++
!   + cpp.et14/
!   |           Expression templates with C++14 standard.
!   + cpp.etp/
!   |           Expression templates with C++11 standard, in a primitive manner 
!   + cpp.oo/
!   |           Typical object-oriented implementation
!   + cpp.valarray/
!   |           Use of STL, valarray
!   + f90/
!   |           low-level fortran code
!   + f90.forall/
!   |           same as f90, but use of forall statement 
!   + f08.concurrent/
!   |           same as f90, but use of do concurrent construct in Fortran2008


## How to compile
1. As an example, in the case of `cpp/` we explain a way of compiling the program. Change directory

In [ ]:
!cd cpp/


2. Make

In [ ]:
## For GNU
!make


The code is successfully compiled by
  * GNU (8.5.0) on x86-64 systems

## Description
All of them calculate a completely identical kernel composed of several multiplication-and-add, but the implementations are different. As a result, the performance could be different. If doing the calculations, you can find `GFLOP/s` in the standard output. Compare such statistical information b/w different kinds of implementation.  

## Exercise 
1.  Compare `GFLOP/s` b/w different kinds of implementation. An expected result on performance (`FLOP/s`) is: `c = cpp = f90 = f90.forall = f08.concurrent = cpp.et14 = cpp.etp = cpp.valarray >> cpp.oo`. Depending on a kind of compiler, the above rank may change.
2. Why are the kernels considered to be **compute-bound** one?

## Reference 
* Yukihiro Ota, Tasunobu Kokubo, and Takaaki Noguchi, "A Performance Analysis of Evaluating Polynomials with 
Expression Templates in Supercomputer K", HPCI Research Report, hp130038 [http://www.hpci-office.jp/annex/resrep/](http://www.hpci-office.jp/annex/resrep/) (and references therein).